In [ ]:
%load_ext autoreload
%autoreload 3

In [ ]:
import pandas as pd
from public_power_backend.constants import OUTPUT_DIR, PUDL_DATA_YEAR

In [ ]:
ious_df = pd.read_parquet(OUTPUT_DIR / "data_warehouse/pudl_utilities.parquet")

In [ ]:
ious_df

In [ ]:
ious_df = ious_df[["utility_name_eia", "utility_id_eia", "utility_id_ferc1", "utility_id_pudl", "state"]]

In [ ]:
ious_df.head(1)

Add on manual mapping to market type.

In [ ]:
manual_map_df = pd.read_csv("regulatory_mapping.csv")

In [ ]:
manual_map_df = manual_map_df.rename(columns={"market status": "market_status", "state_abbreviation": "state"})

In [ ]:
ious_df = ious_df.merge(manual_map_df[["state", "regulator_name", "market_status"]], how="left", on="state")

Get RTOs

In [ ]:
rto_df = pd.read_parquet(
        "s3://pudl.catalyst.coop/stable/core_eia861__yearly_utility_data_rto.parquet",
        dtype_backend="pyarrow",
)
rto_df = rto_df[
    (rto_df.report_date.dt.year >= PUDL_DATA_YEAR)
    & (rto_df.report_date.dt.year < PUDL_DATA_YEAR + 1)
]
rto_df["rtos_of_operation"] = rto_df["rtos_of_operation"].astype("string")

# One-hot encode the rtos_of_operation column into booleans
rtos_dummies = pd.get_dummies(rto_df["rtos_of_operation"], dtype=bool, prefix="in")
rtos_dummies["utility_id_eia"] = rto_df["utility_id_eia"]
# Aggregate by utility_id_eia (any True means the utility operates there)
rtos_wide = rtos_dummies.groupby("utility_id_eia").any().reset_index()
rtos_wide = rtos_wide.drop_duplicates()
assert rtos_wide["utility_id_eia"].is_unique, (
    "utility_id_eia is not unique in EIA RTOs table."
)

In [ ]:
rtos_wide

In [ ]:
reg_con_df = ious_df.merge(rtos_wide,
                        how="left",
                        on="utility_id_eia",
                        validate="1:1")

In [ ]:
rto_cols = ["in_caiso", "in_ercot", "in_isone", "in_miso", "in_nyiso", "in_other", "in_pjm", "in_spp"]
reg_con_df["no_rto_info"] = reg_con_df[rto_cols].isna().all(axis=1)

In [ ]:
reg_con_df

In [ ]:
reg_con_df.to_csv("reg_con.csv")